# Task 2: SelfCheckGPT (reproduction)

**Paper:** Manakul, Liusie, Gales, *SelfCheckGPT: Zero-Resource Black-Box Hallucination Detection*, arXiv:2303.08896.

**Idea.** If the model really knows a fact, asking it again (with random sampling) gives the same fact. If it is making
things up, the samples disagree with each other and with the original answer. So: sample N = 20 more answers and measure
how much they disagree with the greedy answer from Task 1.

* **SelfCheck-BERTScore:** for each sample, the best BERTScore-F1 between the answer and a sentence of that sample
  (`roberta-large`). Score = 1 - average over the 20 samples. High = samples say something different.
* **SelfCheck-NLI:** an NLI model (`microsoft/deberta-large-mnli`) reads (sample, answer) and gives P(contradiction),
  using only the entailment and contradiction outputs, as in the paper. Score = average over the 20 samples.

Our answers are almost always one sentence, so we score the whole answer as one sentence.

Needs `outputs/task1/` from Task 1. Runtime: T4 GPU, about 40 minutes (mostly sampling). Outputs: `outputs/task2/`.

In [ ]:
# save everything to google drive so task 2 and task 3 can read it later
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/project7/outputs', exist_ok=True)
os.chdir('/content/drive/MyDrive/project7')

!pip install -q bert-score

In [ ]:
import json, re
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

SEED = 42
MODEL = "Qwen/Qwen2.5-3B-Instruct"     # must be the same model and prompt as task 1
SYSTEM = "Answer the question in one short sentence."
MAX_NEW = 64
N = 20                                  # samples per question, as in the assignment
Q_PER_BATCH = 8                         # 8 questions x 20 samples per generate call
NLI_MODEL = "microsoft/deberta-large-mnli"
OUT = "outputs/task2"
os.makedirs(OUT, exist_ok=True)
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > T4 GPU"

feat = pd.read_csv("outputs/task1/task1_features.csv")
feat["answer"] = feat["answer"].fillna("")
print(len(feat), "questions from task 1")

## Sample 20 answers per question (temperature 1.0)
Plain sampling from the model's distribution (no top-k / top-p). Samples are appended to `samples.jsonl` after every batch,
so if Colab disconnects, run the notebook again and it continues where it stopped. Each batch has its own fixed seed.

In [ ]:
sample_file = f"{OUT}/samples.jsonl"
samples = {}
if os.path.exists(sample_file):
    for line in open(sample_file):
        try:
            r = json.loads(line)
            samples[r["id"]] = r["samples"]
        except json.JSONDecodeError:
            pass                                  # a half-written line from a crash
todo = [int(i) for i in feat["id"] if int(i) not in samples]
print(len(samples), "questions already sampled,", len(todo), "to go")

if todo:
    tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16)
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16)
    model = model.to("cuda").eval()
    q_by_id = dict(zip(feat["id"].astype(int), feat["question"]))

    for b in range(0, len(todo), Q_PER_BATCH):
        ids = todo[b:b + Q_PER_BATCH]
        prompts = [tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": q_by_id[i]}],
                                           tokenize=False, add_generation_prompt=True) for i in ids]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        torch.manual_seed(SEED + ids[0])
        with torch.no_grad():
            out = model.generate(**enc, do_sample=True, temperature=1.0, top_p=1.0, top_k=0, repetition_penalty=1.0,
                                 max_new_tokens=MAX_NEW, num_return_sequences=N, pad_token_id=tok.pad_token_id)
        texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        with open(sample_file, "a") as f:
            for k, i in enumerate(ids):           # generate returns the N samples of each question together
                samples[i] = [t.strip() for t in texts[k * N:(k + 1) * N]]
                f.write(json.dumps({"id": i, "samples": samples[i]}) + "\n")
        print(f"{len(samples)}/{len(feat)}", end="\r")
    del model
    torch.cuda.empty_cache()

i0 = int(feat["id"][0])
print("\nexample:", feat["question"][0], "\nanswer:", feat["answer"][0], "\nsamples:", samples[i0][:3])

## SelfCheck-BERTScore

In [ ]:
from bert_score import BERTScorer
scorer = BERTScorer(model_type="roberta-large", lang="en", rescale_with_baseline=True, device="cuda")

def sentences(text):
    return [s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s]

# score every (answer, sample sentence) pair once
pairs = list({(a, s) for i, a in zip(feat["id"], feat["answer"]) for x in samples[int(i)] for s in sentences(x)})
f1 = scorer.score([p[0] for p in pairs], [p[1] for p in pairs], batch_size=64)[2].numpy()
f1 = dict(zip(pairs, f1))

def selfcheck_bert(a, sample_list):
    best = [max(f1[(a, s)] for s in sentences(x)) for x in sample_list if sentences(x)]  # best sentence of each sample
    return 1 - np.mean(best) if best else np.nan

feat["selfcheck_bert"] = [selfcheck_bert(a, samples[int(i)]) for i, a in zip(feat["id"], feat["answer"])]
del scorer
torch.cuda.empty_cache()

## SelfCheck-NLI
The premise is the sample and the hypothesis is the answer, i.e. "does this sample contradict the answer?". The label
order of the model is read from its config.

In [ ]:
nli_tok = AutoTokenizer.from_pretrained(NLI_MODEL)
nli = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to("cuda").eval()
lab = {v.lower(): int(k) for k, v in nli.config.id2label.items()}
E, C = lab["entailment"], lab["contradiction"]
print(nli.config.id2label)

@torch.no_grad()
def p_contradiction(premises, hypotheses, bs=32):
    out = []
    for b in range(0, len(premises), bs):
        enc = nli_tok(premises[b:b + bs], hypotheses[b:b + bs], return_tensors="pt",
                      padding=True, truncation=True, max_length=256).to("cuda")
        logits = nli(**enc).logits[:, [E, C]].float()          # drop "neutral", like the paper
        out += torch.softmax(logits, -1)[:, 1].tolist()
    return np.array(out)

prem, hyp, row = [], [], []
for k, (i, a) in enumerate(zip(feat["id"], feat["answer"])):
    for x in samples[int(i)]:
        if x:
            prem.append(x); hyp.append(a); row.append(k)
pc = p_contradiction(prem, hyp)
feat["selfcheck_nli"] = np.bincount(row, pc, minlength=len(feat)) / np.bincount(row, minlength=len(feat))

## Results and comparison with the paper

In [ ]:
ev = feat.dropna(subset=["selfcheck_bert", "selfcheck_nli"])
ev = ev[ev["n_tokens"] > 0]
y = ev["hallucinated"].values
scores = {"SelfCheck-BERTScore": ev["selfcheck_bert"], "SelfCheck-NLI": ev["selfcheck_nli"]}

table = pd.DataFrame({name: {"AUC-PR": average_precision_score(y, s), "AUROC": roc_auc_score(y, s)}
                      for name, s in scores.items()}).T
table.loc["random"] = [y.mean(), 0.5]
table.to_csv(f"{OUT}/task2_results.csv")
print(table.round(3))

# paper, Table 2, NonFact AUC-PR on WikiBio (GPT-3); random = 72.96 there, so compare the gain over random
paper = {"SelfCheck-BERTScore": 81.96, "SelfCheck-NLI": 92.50}
cmp = pd.DataFrame({"ours AUC-PR": table.loc[list(paper), "AUC-PR"] * 100,
                    "ours gain over random": (table.loc[list(paper), "AUC-PR"] - y.mean()) * 100,
                    "paper AUC-PR": pd.Series(paper),
                    "paper gain over random": pd.Series(paper) - 72.96})
cmp.to_csv(f"{OUT}/task2_vs_paper.csv")
print(cmp.round(1))

for name, s in scores.items():
    p, r, _ = precision_recall_curve(y, s)
    plt.plot(r, p, label=name)
plt.axhline(y.mean(), ls="--", c="gray", label="random")
plt.xlabel("recall"); plt.ylabel("precision"); plt.title("Task 2 on TruthfulQA"); plt.legend()
plt.savefig(f"{OUT}/task2_pr.png", dpi=150); plt.show()

feat.to_csv(f"{OUT}/task2_features.csv", index=False)

## A few examples (for the report)
Caught = hallucinated with a high NLI score. Missed = hallucinated but the samples agree with the answer (the model is
consistently wrong). False alarm = accurate answer but the samples disagree.

In [ ]:
def show(rows, title):
    print("=" * 20, title)
    for _, r in rows.iterrows():
        print(f"Q: {r.question}\nA: {r.answer}\n   NLI={r.selfcheck_nli:.2f}  BERT={r.selfcheck_bert:.2f}  "
              f"sample: {samples[int(r.id)][0]}\n")

show(ev[ev.hallucinated == 1].nlargest(3, "selfcheck_nli"), "caught")
show(ev[ev.hallucinated == 1].nsmallest(3, "selfcheck_nli"), "missed")
show(ev[ev.hallucinated == 0].nlargest(3, "selfcheck_nli"), "false alarm")